# 🛒 E-Commerce Sales Analysis — EDA
**Author:** Priyanshi Chauhan  
**Dataset:** UCI E-Commerce Data (via Kaggle)  
**Tools:** Python, Pandas, Matplotlib, Seaborn

## 1. Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Blues_d')

import os
os.makedirs('images', exist_ok=True)

print('Libraries loaded successfully!')

## 2. Load & Explore Data

In [ ]:
df = pd.read_csv('data.csv', encoding='ISO-8859-1')

print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
df.head()

In [ ]:
print('--- Dataset Info ---')
df.info()
print('\n--- Missing Values ---')
print(df.isnull().sum())
print('\n--- Basic Stats ---')
df.describe()

## 3. Data Cleaning

In [ ]:
# Drop rows with missing CustomerID
df.dropna(subset=['CustomerID'], inplace=True)

# Remove cancelled orders (negative quantities) and invalid prices
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]

# Remove rows with missing descriptions
df.dropna(subset=['Description'], inplace=True)

# Create Revenue column
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# Parse dates and extract time features
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Month'] = df['InvoiceDate'].dt.to_period('M')
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
df['Hour'] = df['InvoiceDate'].dt.hour

print(f'Clean dataset shape: {df.shape}')
print(f'Total Revenue: £{df["Revenue"].sum():,.2f}')
print(f'Unique Customers: {df["CustomerID"].nunique()}')
print(f'Unique Products: {df["Description"].nunique()}')
print(f'Countries: {df["Country"].nunique()}')

## 4. Analysis & Visualizations

### 4.1 Monthly Revenue Trend

In [ ]:
monthly_revenue = df.groupby('Month')['Revenue'].sum()

fig, ax = plt.subplots(figsize=(12, 5))
monthly_revenue.plot(kind='line', marker='o', linewidth=2.5, color='steelblue', ax=ax)
ax.fill_between(range(len(monthly_revenue)), monthly_revenue.values, alpha=0.1, color='steelblue')
ax.set_title('Monthly Revenue Trend (2010–2011)', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Revenue (£)', fontsize=11)
ax.set_xticks(range(len(monthly_revenue)))
ax.set_xticklabels([str(m) for m in monthly_revenue.index], rotation=45, ha='right')
plt.tight_layout()
plt.savefig('images/monthly_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

peak_month = monthly_revenue.idxmax()
print(f'Peak Month: {peak_month} — £{monthly_revenue.max():,.2f}')

### 4.2 Top 10 Countries by Revenue

In [ ]:
top_countries = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#1F4E8F' if i == 0 else '#5B9BD5' for i in range(len(top_countries))]
top_countries.plot(kind='bar', color=colors, ax=ax, edgecolor='white')
ax.set_title('Top 10 Countries by Revenue', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Country', fontsize=11)
ax.set_ylabel('Revenue (£)', fontsize=11)
ax.set_xticklabels(top_countries.index, rotation=30, ha='right')
for i, v in enumerate(top_countries):
    ax.text(i, v + 1000, f'£{v/1000:.0f}K', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('images/top_countries.png', dpi=150, bbox_inches='tight')
plt.show()

print(top_countries)

### 4.3 Top 10 Products by Sales Volume

In [ ]:
top_products = df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 6))
top_products.plot(kind='barh', color='coral', ax=ax, edgecolor='white')
ax.set_title('Top 10 Products by Sales Volume', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Total Quantity Sold', fontsize=11)
ax.set_ylabel('')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('images/top_products.png', dpi=150, bbox_inches='tight')
plt.show()

print(top_products)

### 4.4 Revenue by Day of Week

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Sunday']
day_revenue = df.groupby('DayOfWeek')['Revenue'].sum().reindex(day_order)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2ecc71' if v == day_revenue.max() else '#95d5b2' for v in day_revenue]
day_revenue.plot(kind='bar', color=colors, ax=ax, edgecolor='white')
ax.set_title('Revenue by Day of Week', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Day', fontsize=11)
ax.set_ylabel('Revenue (£)', fontsize=11)
ax.set_xticklabels(day_order, rotation=30, ha='right')
plt.tight_layout()
plt.savefig('images/day_of_week.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Best day: {day_revenue.idxmax()} — £{day_revenue.max():,.2f}')

## 5. Summary of Findings

In [ ]:
print('=' * 50)
print('        KEY FINDINGS SUMMARY')
print('=' * 50)
print(f'Total Revenue:       £{df["Revenue"].sum():>12,.2f}')
print(f'Total Transactions:  {df.shape[0]:>12,}')
print(f'Unique Customers:    {df["CustomerID"].nunique():>12,}')
print(f'Unique Products:     {df["Description"].nunique():>12,}')
print(f'Countries Served:    {df["Country"].nunique():>12,}')
print('-' * 50)
print(f'Peak Month:          {str(monthly_revenue.idxmax()):>12}')
print(f'Top Country:         {top_countries.index[0]:>12}')
print(f'Top Product:         {top_products.index[0][:20]:>20}')
print(f'Best Day:            {day_revenue.idxmax():>12}')
print('=' * 50)